# Capítulo 5 · Laboratório 5: o scanner de padrões gráficos
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**. Na primeira execução, o notebook baixa os dados do Yahoo Finance na sua máquina e guarda uma cópia local (`dados_cap05.pkl`), que as execuções seguintes reaproveitam. O provedor corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. O computador acha topos e fundos, traça retas e dá nome às figuras: OCO, topos e fundos duplos, triângulos, cunhas, retângulos, bandeiras, xícaras e outros. Depois mede, nas ações da B3, quantos chegaram ao objetivo e quantos falharam. Regras simplificadas de propósito: mudar a tolerância muda a lista.

In [ ]:
!pip install -q yfinance

In [ ]:
import logging
import os
import numpy as np
import pandas as pd
import yfinance as yf

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
FIM = "2026-10-01"                      # dados até 30/09/2026
CACHE = "dados_cap05.pkl"
pc = lambda x, k=0: f"{x:.{k}%}".replace(".", ",")
nr = lambda x, k=2: f"{x:,.{k}f}".replace(",", "X").replace(".", ",").replace("X", ".")
ACOES = ["PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3", "B3SA3", "ABEV3", "WEGE3", "RENT3", "SUZB3", "GGBR4", "ITSA4",
         "AXIA3", "PRIO3", "RADL3", "LREN3", "BPAC11", "SBSP3", "EQTL3", "VIVT3", "CSNA3", "CSAN3", "RAIL3", "MGLU3",
         "USIM5", "KLBN11", "TOTS3", "HYPE3", "ENEV3", "BRAP4", "CYRE3", "EMBJ3", "UGPA3", "CMIG4", "GOAU4"]
MUSEU = ["VALE5", "TMAR5", "ARCZ6", "LLXL3", "GOLL4", "ELET3"]     # códigos da 3ª edição que sumiram


def baixa(t, ini="2015-01-01"):
    d = yf.download(t, start=ini, end=FIM, auto_adjust=False, progress=False)
    if isinstance(d.columns, pd.MultiIndex):
        d.columns = d.columns.get_level_values(0)
    return d.dropna(subset=["Close"])


def confere(baixados):
    """O Yahoo devolve uma tabela vazia quando falha: para aqui, com uma mensagem clara, sem gravar a cópia local."""
    for nome, x in baixados.items():
        if len(x) == 0:
            raise RuntimeError(f"Não consegui baixar {nome} do Yahoo Finance. O provedor pode estar fora do ar ou"
                               " limitando pedidos. Espere alguns minutos e rode a célula de novo.")


D = pd.read_pickle(CACHE) if os.path.exists(CACHE) else {}
for t in [a + ".SA" for a in ACOES] + ["^BVSP"]:
    if t not in D:
        D[t] = baixa(t, "2000-01-01" if t == "^BVSP" else "2015-01-01")
confere({"^BVSP": D["^BVSP"]})
pd.to_pickle(D, CACHE)

## Parte 1: dados e o museu de tickers

In [ ]:
def limpa(d, indice=False):
    ok = (d[["Open", "High", "Low", "Close"]] > 0).all(axis=1) & (d["High"] >= d["Low"])
    if not indice:
        ok &= d["Volume"] > 0
    return d[ok].copy()


B = {a: limpa(D[a + ".SA"]) for a in ACOES}
B["IBOV"] = limpa(D["^BVSP"], True).loc["2015-01-01":]
print(f"Parte 1. {len(ACOES)} ações e o Ibovespa, de 2015 a 30/09/2026 (preço de tela, sem ajuste de proventos).")
print("   Museu de tickers (códigos da 3ª edição):", ", ".join(MUSEU),
      "- o provedor gratuito não tem mais esses códigos. Estudo antigo precisa conferir o código antes de rodar.")

## Parte 2: pivôs pelo ziguezague

In [ ]:
def zigzag(d, limite=0.05):
    """Topos e fundos pela máxima e pela mínima: um topo só vale depois que o preço cai 'limite' a partir dele
    (e um fundo, depois que sobe 'limite'). Devolve [(posição, preço, 'T' ou 'F')]."""
    h, l = d["High"].to_numpy(), d["Low"].to_numpy()
    piv, tend, imax, imin = [], 0, 0, 0
    for i in range(1, len(h)):
        if h[i] > h[imax]:
            imax = i
        if l[i] < l[imin]:
            imin = i
        if tend >= 0 and l[i] < h[imax] * (1 - limite) and (not piv or piv[-1][2] == "F"):
            piv.append((imax, h[imax], "T"))
            tend = -1
            imin = imax + 1 + int(np.argmin(l[imax + 1:i + 1])) if imax < i else i   # o fundo conta depois do topo
        elif tend <= 0 and h[i] > l[imin] * (1 + limite) and (not piv or piv[-1][2] == "T"):
            piv.append((imin, l[imin], "F"))
            tend = 1
            imax = imin + 1 + int(np.argmax(h[imin + 1:i + 1])) if imin < i else i
    return piv


Z5 = {a: zigzag(d, 0.05) for a, d in B.items()}
Z8 = {a: zigzag(d, 0.08) for a, d in B.items()}
print(f"\nParte 2. Pivôs com reversão de 5%: {sum(len(z) for z in Z5.values()):,} em {len(B)} séries.".replace(",", "."))

PREF = {"ITUB4": 0.3, "BPAC11": 0.3, "SBSP3": 0.3}      # ações escolhidas pelo autor para os exemplos


def recente(d, i):
    return 0.2 if d.index[i].year >= 2023 else (0.1 if d.index[i].year >= 2021 else 0.0)


def linha(x0, y0, x1, y1):
    m = (y1 - y0) / (x1 - x0)
    return lambda x: y0 + m * (np.asarray(x) - x0)


def alvo_queda(nivel, altura):
    """Objetivo de queda em porcentagem (Dica do Flávio, Cap. 5): a figura cai a mesma porcentagem da sua altura,
    medida a partir do topo do padrão. Nunca fica negativo e é mais conservador que subtrair a altura em pontos."""
    return nivel * (1 - altura / (nivel + altura))

## Parte 3: OCO e OCO invertido

In [ ]:
def oco(a, d, piv):
    """OCO: topo, fundo, topo mais alto (cabeça), fundo, topo. OCOI: o espelho."""
    c, h, l = d["Close"].to_numpy(), d["High"].to_numpy(), d["Low"].to_numpy()
    out = []
    for k in range(len(piv) - 4):
        p = piv[k:k + 5]
        tipo = "OCO" if p[0][2] == "T" else "OCOI"
        (i1, o1, _), (j1, n1, _), (i2, cab, _), (j2, n2, _), (i3, o2, _) = p
        s = 1 if tipo == "OCO" else -1
        prof = (cab - max(o1, o2)) if s > 0 else (min(o1, o2) - cab)
        if prof < 0.03 * cab:
            continue                                  # a cabeça precisa passar dos ombros em pelo menos 3%
        if abs(o2 / o1 - 1) > 0.05 or not 20 <= i3 - i1 <= 150 or i1 < 60:
            continue
        antes = d["Close"].iloc[i1 - 60:i1]
        if s > 0 and antes.min() > o1 * 0.87 or s < 0 and antes.max() < o1 * 1.13:
            continue                                  # sem tendência anterior para reverter
        pesc = linha(j1, n1, j2, n2)
        quebra = None
        for t in range(i3 + 1, min(i3 + 30, len(d))):
            if s * (c[t] - pesc(t)) < 0:
                quebra = t
                break
            if s * (h[t] if s > 0 else l[t]) > s * cab:
                break
        if quebra is None:
            continue
        alvo_pts = pesc(quebra) - s * (s * (cab - pesc(i2)))
        alvo = alvo_queda(pesc(quebra), cab - pesc(i2)) if s > 0 else alvo_pts
        fim = min(quebra + 120, len(d))
        atingiu = (l[quebra:fim].min() <= alvo) if s > 0 else (h[quebra:fim].max() >= alvo)
        at_pts = (l[quebra:fim].min() <= alvo_pts) if s > 0 else (h[quebra:fim].max() >= alvo_pts)
        nota = 1 - abs(o2 / o1 - 1) * 10 - abs(n2 / n1 - 1) * 5 + 0.5 * at_pts + PREF.get(a, 0) + recente(d, i1)
        out.append(dict(tipo=tipo, ativo=a, i=[i1, j1, i2, j2, i3], quebra=quebra, alvo=alvo, atingiu=bool(atingiu),
                        alvo_pts=alvo_pts, atingiu_pts=bool(at_pts), lado=-s,
                        nota=nota, inicio=d.index[i1], fim=d.index[quebra]))
    return out

## Parte 4: topo duplo, fundo duplo, Adão e Eva

In [ ]:
def largura(d, i, nivel, s):
    """Quantos pregões, perto do pivô, ficaram a menos de 3% do topo (ou do fundo): Adão é fino, Eva é larga."""
    jan = d.iloc[max(i - 15, 0):i + 16]
    return int((jan["High"] >= nivel * 0.97).sum()) if s > 0 else int((jan["Low"] <= nivel * 1.03).sum())


def duplos(a, d, piv):
    c, h, l = d["Close"].to_numpy(), d["High"].to_numpy(), d["Low"].to_numpy()
    out = []
    for k in range(len(piv) - 2):
        (i1, p1, t1), (j, m, _), (i2, p2, _) = piv[k:k + 3]
        s = 1 if t1 == "T" else -1
        if abs(p2 / p1 - 1) > 0.03 or not 20 <= i2 - i1 <= 130 or i1 < 60:
            continue
        if abs(p1 - m) / p1 < 0.08:
            continue                                  # o vale (ou o pico) do meio precisa ter pelo menos 8%
        antes = d["Close"].iloc[i1 - 60:i1]
        if s > 0 and antes.min() > p1 * 0.87 or s < 0 and antes.max() < p1 * 1.13:
            continue
        ext = max(p1, p2) if s > 0 else min(p1, p2)
        quebra = None
        for t in range(i2 + 1, min(i2 + 40, len(d))):
            if s * (c[t] - m) < 0:
                quebra = t
                break
            if s > 0 and h[t] > ext * 1.02 or s < 0 and l[t] < ext * 0.98:
                break
        if quebra is None:
            continue
        alvo_pts = m - s * abs(ext - m)
        alvo = alvo_queda(m, abs(ext - m)) if s > 0 else alvo_pts
        fim = min(quebra + 120, len(d))
        atingiu = (l[quebra:fim].min() <= alvo) if s > 0 else (h[quebra:fim].max() >= alvo)
        at_pts = (l[quebra:fim].min() <= alvo_pts) if s > 0 else (h[quebra:fim].max() >= alvo_pts)
        w1, w2 = largura(d, i1, p1, s), largura(d, i2, p2, s)
        nota = 1 - abs(p2 / p1 - 1) * 10 + 0.5 * at_pts + PREF.get(a, 0) + recente(d, i1)
        out.append(dict(tipo="topo duplo" if s > 0 else "fundo duplo", ativo=a, i=[i1, j, i2], quebra=quebra,
                        alvo=alvo, atingiu=bool(atingiu), alvo_pts=alvo_pts, atingiu_pts=bool(at_pts), lado=-s, nota=nota, larg=(w1, w2), inicio=d.index[i1],
                        fim=d.index[quebra]))
        if s < 0 and min(w1, w2) <= 3 and max(w1, w2) >= 8:
            out.append(dict(out[-1], tipo="Adão e Eva", nota=nota + 0.1 * abs(w1 - w2)))
    return out

## Parte 5: triângulos, cunhas, retângulos e alargamento

In [ ]:
def ajusta(xs, ys):
    m, b = np.polyfit(xs, ys, 1)
    return m, b, np.max(np.abs(ys - (m * np.asarray(xs) + b)) / np.asarray(ys))


def caixas(a, d, piv, max_desvio=0.015, plano=0.0004):
    """Sequências de 5 a 7 pivôs alternados (pelo menos 2 topos e 2 fundos): uma reta pelos topos e outra pelos
    fundos. A inclinação de cada reta (em % do preço por pregão) diz o nome do padrão. Depois, o rompimento:
    o primeiro fechamento fora das retas em até 20 pregões depois do último pivô."""
    c, h, l = d["Close"].to_numpy(), d["High"].to_numpy(), d["Low"].to_numpy()
    out = []
    for n in (5, 6, 7):
        for k in range(len(piv) - n + 1):
            p = piv[k:k + n]
            x0, x1 = p[0][0], p[-1][0]
            if not 20 <= x1 - x0 <= 160 or x0 < 60:
                continue
            ts = [(i, v) for i, v, t in p if t == "T"]
            fs = [(i, v) for i, v, t in p if t == "F"]
            mt_, bt, et = ajusta(*zip(*ts))
            mf, bf, ef = ajusta(*zip(*fs))
            if et > max_desvio or ef > max_desvio:
                continue
            ref = np.mean([v for _, v, _ in p])
            st, sf = mt_ / ref, mf / ref
            sup = lambda x: mt_ * x + bt
            inf = lambda x: mf * x + bf
            xs = np.arange(x0, x1 + 1)
            if (c[xs] > sup(xs) * 1.01).any() or (c[xs] < inf(xs) * 0.99).any():
                continue                              # algum fechamento fura as retas: não é uma figura limpa
            alt0, alt1 = sup(x0) - inf(x0), sup(x1) - inf(x1)
            if alt0 <= 0 or alt1 <= 0:
                continue
            if abs(st) < plano and abs(sf) < plano:
                nome = "retângulo"
            elif abs(st) < plano and sf > plano:
                nome = "triângulo ascendente"
            elif abs(sf) < plano and st < -plano:
                nome = "triângulo descendente"
            elif st < -plano and sf > plano:
                nome = "triângulo simétrico"
            elif st < -plano and sf < -plano and alt1 < alt0 * 0.75:
                nome = "cunha descendente"
            elif st > plano and sf > plano and alt1 < alt0 * 0.75:
                nome = "cunha ascendente"
            elif st > plano and sf < -plano:
                nome = "alargamento"
            else:
                continue
            if nome == "retângulo" and not 0.05 <= alt0 / ref <= 0.25:
                continue
            quebra, lado = None, 0
            for t in range(x1 + 1, min(x1 + 20, len(d))):
                if c[t] > sup(t) * 1.005:
                    quebra, lado = t, 1
                    break
                if c[t] < inf(t) * 0.995:
                    quebra, lado = t, -1
                    break
            if quebra is None:
                continue
            altura = max(alt0, alt1)
            nivel = sup(quebra) if lado > 0 else inf(quebra)
            alvo_pts = nivel + lado * altura
            alvo = alvo_pts if lado > 0 else alvo_queda(nivel, altura)
            fim = min(quebra + 120, len(d))
            atingiu = (h[quebra:fim].max() >= alvo) if lado > 0 else (l[quebra:fim].min() <= alvo)
            at_pts = (h[quebra:fim].max() >= alvo_pts) if lado > 0 else (l[quebra:fim].min() <= alvo_pts)
            # falha (padrão "busted", de Bulkowski): anda menos de 10% a favor e fecha do outro lado da figura
            mais = (h[quebra:fim].max() / c[quebra] - 1) if lado > 0 else (1 - l[quebra:fim].min() / c[quebra])
            outro = (c[quebra:fim] < inf(np.arange(quebra, fim)) * 0.995) if lado > 0 else \
                (c[quebra:fim] > sup(np.arange(quebra, fim)) * 1.005)
            falhou = bool(mais < 0.10 and outro.any())
            tend = c[x0] / c[x0 - 60] - 1
            nota = 1 - (et + ef) * 20 + 0.1 * n + 0.5 * at_pts + PREF.get(a, 0) + recente(d, x0)
            out.append(dict(tipo=nome, ativo=a, i=[i for i, _, _ in p], quebra=quebra, lado=lado, alvo=alvo,
                            atingiu=bool(atingiu), alvo_pts=alvo_pts, atingiu_pts=bool(at_pts), falhou=falhou, tend=tend, linhas=(mt_, bt, mf, bf), nota=nota,
                            inicio=d.index[x0], fim=d.index[quebra]))
    return out

## Parte 6: bandeiras e flâmulas

In [ ]:
def bandeiras(a, d):
    """Mastro: alta (ou queda) de pelo menos 12% em até 12 pregões. Pano: 5 a 20 pregões de pausa que devolvem menos
    da metade do mastro. Bandeira: retas paralelas contra a tendência. Flâmula: retas que se fecham."""
    c, h, l = d["Close"].to_numpy(), d["High"].to_numpy(), d["Low"].to_numpy()
    out, ult = [], -50
    for t in range(80, len(d) - 30):
        if t - ult < 20:
            continue
        for s in (1, -1):
            base = l[t - 12:t + 1].min() if s > 0 else h[t - 12:t + 1].max()
            topo = h[t] if s > 0 else l[t]
            if s * (topo / base - 1) < 0.12 or (s > 0 and h[t] < h[t - 12:t + 6].max()) or \
                    (s < 0 and l[t] > l[t - 12:t + 6].min()):
                continue
            mastro = abs(topo - base)
            for dur in range(5, 21):
                pano = slice(t + 1, t + 1 + dur)
                if s > 0 and (h[pano].max() > topo * 1.01 or l[pano].min() < topo - 0.5 * mastro):
                    break
                if s < 0 and (l[pano].min() < topo * 0.99 or h[pano].max() > topo + 0.5 * mastro):
                    break
                q = t + 1 + dur
                lim = h[t + 1:q].max() if s > 0 else l[t + 1:q].min()
                if q < len(d) and s * (c[q] - lim) > 0:
                    xs = np.arange(t + 1, q)
                    mh, _ = np.polyfit(xs, h[t + 1:q], 1)
                    ml, _ = np.polyfit(xs, l[t + 1:q], 1)
                    paralelas = abs(mh - ml) < 0.25 * max(abs(mh), abs(ml), 1e-9) and s * mh < 0
                    fechando = mh < 0 < ml
                    if not (paralelas or fechando):
                        break
                    alvo_pts = lim + s * mastro
                    alvo = alvo_pts if s > 0 else alvo_queda(lim, mastro)
                    fim = min(q + 60, len(d))
                    atingiu = (h[q:fim].max() >= alvo) if s > 0 else (l[q:fim].min() <= alvo)
                    at_pts = (h[q:fim].max() >= alvo_pts) if s > 0 else (l[q:fim].min() <= alvo_pts)
                    nota = 1 + 0.5 * at_pts + PREF.get(a, 0) + recente(d, t) + (0.2 if dur >= 7 else 0)
                    out.append(dict(tipo="bandeira" if paralelas else "flâmula", ativo=a,
                                    i=[int(np.where(l[t - 12:t + 1] == base)[0][0]) + t - 12 if s > 0 else
                                       int(np.where(h[t - 12:t + 1] == base)[0][0]) + t - 12, t, q - 1],
                                    quebra=q, lado=s, alvo=alvo, atingiu=bool(atingiu), nota=nota,
                                    alvo_pts=alvo_pts, atingiu_pts=bool(at_pts),
                                    inicio=d.index[t - 12], fim=d.index[q]))
                    ult = t
                    break
    return out

## Parte 7: outros padrões

In [ ]:
def movimento_medido(a, d, piv):
    out = []
    h = d["High"].to_numpy()
    for k in range(len(piv) - 3):
        (i0, f0, t0), (i1, t1, _), (i2, f1, _), (i3, t2, _) = piv[k:k + 4]
        if t0 != "F":
            continue
        perna1, corr = t1 - f0, (t1 - f1) / (t1 - f0)
        if perna1 / f0 < 0.15 or not 0.3 <= corr <= 0.62:
            continue
        razao = (t2 - f1) / perna1
        if not 0.85 <= razao <= 1.2:
            continue
        nota = 1 - abs(razao - 1) * 3 + PREF.get(a, 0) + recente(d, i0)
        out.append(dict(tipo="movimento medido", ativo=a, i=[i0, i1, i2, i3], razao=razao, correcao=corr,
                        alvo=f1 + perna1, nota=nota, inicio=d.index[i0], fim=d.index[i3]))
    return out


def canais(a, d, n=160):
    out = []
    lc = np.log(d["Close"].to_numpy())
    lh, ll = np.log(d["High"].to_numpy()), np.log(d["Low"].to_numpy())
    for s0 in range(60, len(d) - n, 10):
        xs = np.arange(s0, s0 + n)
        m, b = np.polyfit(xs, lc[xs], 1)
        res = lc[xs] - (m * xs + b)
        r2 = 1 - res.var() / lc[xs].var()
        if r2 < 0.85 or abs(np.expm1(m * 21)) < 0.02:
            continue
        cima, baixo = (lh[xs] - (m * xs + b)).max(), (ll[xs] - (m * xs + b)).min()
        if cima - baixo > 0.30:
            continue
        toq_c = np.where(lh[xs] - (m * xs + b) > cima - 0.015)[0]
        toq_b = np.where(ll[xs] - (m * xs + b) < baixo + 0.015)[0]
        conta = lambda v: int((np.diff(v) > 8).sum() + 1) if len(v) else 0
        tc, tb = conta(toq_c), conta(toq_b)
        if min(tc, tb) < 3:
            continue
        nota = r2 + 0.1 * (tc + tb) + PREF.get(a, 0) + recente(d, s0)
        out.append(dict(tipo="canal", ativo=a, i=[s0, s0 + n - 1], linhas=(m, b, cima, baixo), toques=(tc, tb),
                        nota=nota, inicio=d.index[s0], fim=d.index[s0 + n - 1]))
    return out


def semanal(d):
    return d.resample("W-FRI").agg({"Open": "first", "High": "max", "Low": "min", "Close": "last",
                                    "Volume": "sum"}).dropna()


def fundo_arredondado(a, d):
    w = semanal(d)
    lc = np.log(w["Close"].to_numpy())
    out = []
    for n in (30, 40, 52, 65):
        for s0 in range(0, len(w) - n - 12, 2):
            xs = np.arange(n)
            y = lc[s0:s0 + n]
            cf = np.polyfit(xs, y, 2)
            res = y - np.polyval(cf, xs)
            r2 = 1 - res.var() / y.var()
            v = -cf[1] / (2 * cf[0]) if cf[0] > 0 else -1
            if cf[0] <= 0 or r2 < 0.8 or not 0.3 * n < v < 0.7 * n:
                continue
            borda = w["High"].iloc[s0:s0 + 4].max()
            fundo = w["Low"].iloc[s0:s0 + n].min()
            if borda / fundo - 1 < 0.25 or abs(w["Close"].iloc[s0 + n - 1] / borda - 1) > 0.12:
                continue
            depois = w["Close"].iloc[s0 + n:s0 + n + 12]
            if not (depois > borda).any():
                continue
            nota = r2 + PREF.get(a, 0) + (0.2 if w.index[s0].year >= 2021 else 0)
            out.append(dict(tipo="fundo arredondado", ativo=a, i=[s0, s0 + n - 1], coef=cf, borda=borda, nota=nota,
                            inicio=w.index[s0], fim=w.index[s0 + n - 1], semanal=True))
    return out


def xicaras(a, d, piv):
    """Borda esquerda (topo), xícara em U (queda de 12% a 40%), borda direita perto da esquerda, alça curta
    (recuo de 3% a 15%, até 30 pregões) e rompimento acima das bordas."""
    c, h, l = d["Close"].to_numpy(), d["High"].to_numpy(), d["Low"].to_numpy()
    out = []
    for k in range(len(piv) - 2):
        (i1, b1, t1), (j, f, _), (i2, b2, _) = piv[k:k + 3]
        if t1 != "T" or not 35 <= i2 - i1 <= 200 or i1 < 60:
            continue
        prof = 1 - f / b1
        if not 0.12 <= prof <= 0.40 or abs(b2 / b1 - 1) > 0.05:
            continue
        fundo_u = (c[i1:i2] < f + (b1 - f) / 3).mean()
        if fundo_u < 0.25:
            continue                                  # V agudo, não U
        if c[i1 - 60] > b1 * 0.9:
            continue                                  # sem alta antes da xícara
        teto = max(b1, b2)
        for q in range(i2 + 5, min(i2 + 31, len(d))):
            alca = l[i2:q].min()
            recuo = 1 - alca / b2
            if recuo > 0.15 or recuo > prof / 2:
                break
            if c[q] > teto and recuo >= 0.03:
                alvo = teto + (teto - f)
                fim = min(q + 120, len(d))
                atingiu = h[q:fim].max() >= alvo
                nota = 1 + fundo_u + 0.5 * atingiu - abs(b2 / b1 - 1) * 5 + PREF.get(a, 0) + recente(d, i1)
                out.append(dict(tipo="xícara com alça", ativo=a, i=[i1, j, i2, int(np.argmin(l[i2:q])) + i2],
                                quebra=q, alvo=alvo, atingiu=bool(atingiu), nota=nota, inicio=d.index[i1],
                                fim=d.index[q]))
                break
    return out


def diamantes(a, d, piv):
    """Topos que sobem e depois descem; fundos que descem e depois sobem (alargamento seguido de triângulo)."""
    c = d["Close"].to_numpy()
    out = []
    for k in range(len(piv) - 6):
        p = piv[k:k + 7]
        if p[0][2] != "T":
            continue
        T = [v for _, v, t in p if t == "T"]       # 4 topos
        F = [v for _, v, t in p if t == "F"]       # 3 fundos
        if not (T[0] < T[1] > T[2] > T[3] and F[0] > F[1] < F[2]):
            continue
        if not 30 <= p[-1][0] - p[0][0] <= 160:
            continue
        linha_baixo = linha(p[3][0], F[1], p[5][0], F[2])
        quebra = next((t for t in range(p[-1][0] + 1, min(p[-1][0] + 20, len(d))) if c[t] < linha_baixo(t)), None)
        if quebra is None:
            continue
        nota = 1 + PREF.get(a, 0) + recente(d, p[0][0])
        out.append(dict(tipo="diamante", ativo=a, i=[i for i, _, _ in p], quebra=quebra, nota=nota,
                        inicio=d.index[p[0][0]], fim=d.index[quebra]))
    return out


def solavanco(a, d):
    """Introdução: reta pelas mínimas de 40 a 80 pregões, com inclinação moderada. Solavanco: a máxima se afasta da
    reta pelo menos o dobro do maior afastamento da introdução (e mais de 15%). Fuga: fecha abaixo da reta."""
    c, h, l = d["Close"].to_numpy(), d["High"].to_numpy(), d["Low"].to_numpy()
    out, ult = [], -100
    for s0 in range(60, len(d) - 120, 5):
        if s0 - ult < 60:
            continue
        for n in (40, 60, 80):
            xs = np.arange(s0, s0 + n)
            m, b = np.polyfit(xs, l[xs], 1)
            b += (l[xs] - (m * xs + b)).min()           # a reta passa por baixo de todas as mínimas
            inc = m / l[s0] * 21
            if not 0.01 <= inc <= 0.06:
                continue
            reta = lambda x: m * np.asarray(x) + b
            d1 = (h[xs] - reta(xs)).max()
            seg = np.arange(s0 + n, min(s0 + n + 100, len(d)))
            fura = seg[c[seg] < reta(seg)]
            if not len(fura):
                continue
            q = fura[0]
            topo = s0 + n + int(np.argmax(h[s0 + n:q])) if q > s0 + n else None
            if topo is None:
                continue
            d2 = h[topo] - reta(topo)
            if d2 < 2 * d1 or d2 / reta(topo) < 0.15:
                continue
            fim = min(q + 60, len(d))
            nota = 1 + min(d2 / d1, 4) * 0.1 + PREF.get(a, 0) + recente(d, s0) + (0.3 if a == "BBAS3" else 0)
            out.append(dict(tipo="solavanco e fuga", ativo=a, i=[s0, s0 + n - 1, topo], quebra=int(q),
                            reta=(m, b), d1=d1, d2=d2, queda=1 - l[q:fim].min() / c[q], nota=nota,
                            inicio=d.index[s0], fim=d.index[q]))
            ult = s0
            break
    return out


TODOS = []
for a, d in B.items():
    p5, p8 = Z5[a], Z8[a]
    TODOS += oco(a, d, p5) + oco(a, d, p8) + duplos(a, d, p5) + caixas(a, d, p5) + bandeiras(a, d)
    TODOS += movimento_medido(a, d, p8) + canais(a, d) + fundo_arredondado(a, d) + xicaras(a, d, p8)
    TODOS += diamantes(a, d, p5) + solavanco(a, d)
P = pd.DataFrame(TODOS)
print("\nParte 3 a 7. Padrões encontrados de 2015 a 2026 (mesma figura pode aparecer em janelas vizinhas):")
print(P["tipo"].value_counts().to_string())


def melhores(tipo, n=3, **filtro):
    s = P[P["tipo"] == tipo]
    for k, v in filtro.items():
        s = s[s[k] == v]
    s = s.sort_values("nota", ascending=False)
    s = s.loc[~s[["ativo", "fim"]].astype(str).agg("|".join, axis=1).duplicated()]
    return s.head(n)


print("\n   Melhores exemplos (ativo, início e rompimento) para os gráficos:")
for tp in P["tipo"].unique():
    m = melhores(tp)
    print(f"   {tp}: " + "; ".join(f"{r.ativo} {r.inicio:%d/%m/%Y} a {r.fim:%d/%m/%Y}" for r in m.itertuples()))

In [ ]:
import matplotlib.pyplot as plt

def mostra(tipo, n=0, **filtro):
    """Desenha o n-ésimo melhor exemplo de um padrão: fechamento, pivôs (círculos) e rompimento (linha pontilhada)."""
    r = melhores(tipo, n + 1, **filtro).iloc[n]
    d = B[r["ativo"]]
    q = int(r["quebra"]) if "quebra" in r and r["quebra"] == r["quebra"] else max(r["i"])
    i0, i1 = max(min(r["i"]) - 30, 0), min(q + 60, len(d) - 1)
    w = d.iloc[i0:i1]
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.plot(range(len(w)), w["Close"], color="black", lw=0.9)
    for i in r["i"]:
        ax.plot(i - i0, d["Close"].iloc[i], "o", mfc="none", mec="gray")
    ax.axvline(q - i0, color="gray", ls=":")
    ax.set_title(f"{tipo}: {r['ativo']} de {r['inicio']:%d/%m/%Y} a {r['fim']:%d/%m/%Y}")
    plt.show()

mostra("OCO"); mostra("fundo duplo"); mostra("triângulo ascendente")

## Parte 8: a ficha da B3

In [ ]:
def dedup(s):
    """Uma figura por ativo e data de rompimento."""
    return s.sort_values("nota", ascending=False).drop_duplicates(["ativo", "quebra"])


FICHA = []
for tp in ["OCO", "OCOI", "topo duplo", "fundo duplo", "triângulo ascendente", "triângulo descendente",
           "triângulo simétrico", "retângulo", "cunha ascendente", "cunha descendente", "alargamento",
           "bandeira", "flâmula", "xícara com alça"]:
    s = dedup(P[P["tipo"] == tp])
    if not len(s):
        continue
    lin = {"padrão": tp, "casos": len(s), "objetivo atingido": s["atingiu"].mean()}
    if "lado" in s and s["lado"].notna().any() and tp in ("triângulo simétrico", "retângulo", "alargamento",
                                                           "triângulo ascendente", "triângulo descendente",
                                                           "cunha ascendente", "cunha descendente"):
        lin["rompeu para cima"] = (s["lado"] > 0).mean()
        lin["falhou (busted)"] = s["falhou"].mean()
    FICHA.append(lin)
FICHA = pd.DataFrame(FICHA).set_index("padrão")
print("\nParte 8. Ficha da B3 (35 ações, 2015 a 2026; objetivo em até 120 pregões; regras simplificadas):")
print(FICHA.to_string(formatters={k: pc for k in FICHA.columns if k != "casos"}, na_rep="-"))

# Parte 8b: objetivo de queda em pontos ou em porcentagem (Dica do Flávio)
Q = []
for tp in ["OCO", "topo duplo", "triângulo descendente", "triângulo simétrico", "retângulo", "cunha ascendente",
           "alargamento", "bandeira", "flâmula"]:
    s = dedup(P[(P["tipo"] == tp)])
    s = s[s["lado"] < 0] if "lado" in s else s.iloc[0:0]
    if not len(s):
        continue
    Q.append({"padrão": tp, "quedas": len(s), "objetivo em pontos": s["atingiu_pts"].mean(),
              "objetivo em %": s["atingiu"].mean(), "objetivo em pontos <= 0": int((s["alvo_pts"] <= 0).sum())})
Q = pd.DataFrame(Q).set_index("padrão")
print("\nParte 8b. Rompimentos para baixo: objetivo medido em pontos (altura subtraída) ou em % (mesma porcentagem"
      " da altura, medida a partir do topo):")
print(Q.to_string(formatters={"objetivo em pontos": pc, "objetivo em %": pc}))

## Parte 9: quanto maior a base, maior a alta?

In [ ]:
def bases(d, olhar=60):
    """Rompimento = fechamento acima da máxima dos 20 pregões anteriores. Tamanho da base = há quantos pregões o preço
    não fechava acima desse nível. Mede a maior alta (pela máxima) e o retorno em 'olhar' pregões."""
    c, h = d["Close"].to_numpy(), d["High"].to_numpy()
    lin = []
    ult = -20
    for t in range(260, len(d) - olhar):
        nivel = h[t - 20:t].max()
        if c[t] <= nivel or c[t - 1] > h[t - 21:t - 1].max() or t - ult < 10:
            continue
        acima = np.where(c[:t - 20] > nivel)[0]
        base = t - (acima[-1] if len(acima) else 0)
        lin.append((base, h[t + 1:t + 1 + olhar].max() / c[t] - 1, c[t + olhar] / c[t] - 1))
        ult = t
    return lin


BS = pd.DataFrame([x for a in ACOES for x in bases(B[a])], columns=["base", "maior_alta", "ret60"])
BS["faixa"] = pd.cut(BS["base"], [0, 40, 120, 250, 10_000],
                     labels=["até 40 pregões", "41 a 120", "121 a 250", "mais de 250 (um ano ou mais)"])
TB = BS.groupby("faixa", observed=True).agg(casos=("base", "size"), maior_alta=("maior_alta", "median"),
                                            ret60=("ret60", "median"), positivos=("ret60", lambda x: (x > 0).mean()))
print("\nParte 9. Rompimentos de máxima de 20 pregões nas 35 ações, por tamanho da base (medianas, 60 pregões depois):")
print(TB.to_string(formatters={"maior_alta": lambda x: pc(x, 1), "ret60": lambda x: pc(x, 1), "positivos": pc}))

## Parte 10: volume no rompimento é obrigatório?

In [ ]:
def volume_rompimento(d, olhar=20):
    c, h, v = d["Close"].to_numpy(), d["High"].to_numpy(), d["Volume"].to_numpy()
    vm = pd.Series(v).rolling(20).mean().shift().to_numpy()
    lin, ult = [], -20
    for t in range(60, len(d) - olhar):
        nivel = h[t - 20:t].max()
        if c[t] <= nivel or c[t - 1] > h[t - 21:t - 1].max() or t - ult < 10:
            continue
        falso = (c[t + 1:t + 1 + 10] < nivel).any()
        lin.append((v[t] / vm[t], c[t + olhar] / c[t] - 1, falso))
        ult = t
    return lin


VR = pd.DataFrame([x for a in ACOES for x in volume_rompimento(B[a])], columns=["vol_x", "ret20", "falso"])
VR["volume"] = pd.cut(VR["vol_x"], [0, 1, 1.5, 100], labels=["abaixo da média", "1 a 1,5 vez", "mais de 1,5 vez"])
TV = VR.groupby("volume", observed=True).agg(casos=("ret20", "size"), ret20=("ret20", "median"),
                                             positivos=("ret20", lambda x: (x > 0).mean()), falsos=("falso", "mean"))
print("\nParte 10. Volume no dia do rompimento (máxima de 20 pregões) e o que veio depois (20 pregões):")
print(TV.to_string(formatters={"ret20": lambda x: pc(x, 1), "positivos": pc, "falsos": pc}))

## Parte 11: padrões em formação em 30/09/2026

In [ ]:
def em_formacao(a, d, piv, tol=0.03):
    c = d["Close"].iloc[-1]
    achados = []
    if len(piv) >= 3:
        (i1, p1, t1), (j, m, _), (i2, p2, _) = piv[-3:]
        if abs(p2 / p1 - 1) <= tol and i2 - i1 >= 15 and abs(p1 - m) / p1 >= 0.06:
            nome = "fundo duplo" if t1 == "F" else "topo duplo"
            ok = c > m if t1 == "F" else c < m
            achados.append((a, nome, d.index[i1], d.index[i2], m, c, "confirmado" if ok else "aguardando rompimento"))
    return achados


AGORA = pd.DataFrame([x for a in ACOES for x in em_formacao(a, B[a], Z5[a])],
                     columns=["ação", "padrão", "1º pivô", "2º pivô", "linha (R$)", "último (R$)", "situação"])
print("\nParte 11. Topos e fundos duplos nos últimos pivôs (30/09/2026):")
print(AGORA.to_string(index=False, formatters={"1º pivô": lambda x: x.strftime("%d/%m/%Y"),
                                               "2º pivô": lambda x: x.strftime("%d/%m/%Y"),
                                               "linha (R$)": nr, "último (R$)": nr}) if len(AGORA) else "   nenhum")

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique a função `alvo_queda` da Parte 2. Mostre, com o exemplo do capítulo (topo em R$ 10 e linha de pescoço em R$ 4), por que a conta em porcentagem dá R$ 1,60 e a conta em pontos daria R$ -2." Confira com a Parte 8, que imprime a comparação 8b: no OCO, 55% chegaram ao objetivo em porcentagem, contra 49% em pontos; no topo duplo, 64% contra 58%, como no livro. São números do passado: regra tirada daqui precisa do teste fora da amostra (Capítulo 27).
2. **Adaptar:** "Numa célula nova depois da Parte 11, rode `em_formacao` para [ATIVO]: baixe com `baixa`, limpe com `limpa` e ache os pivôs com `zigzag(d, 0.05)`. Não mexa na lista `ACOES` nem na tolerância de 3%." Antes de olhar o ativo novo, rode o notebook inteiro e confira se a ficha da Parte 8 continua com os casos do livro: 141 OCOs, 118 OCOs invertidos, 33 topos duplos e 46 fundos duplos. Topo ou fundo duplo em formação ainda não é padrão: só vale depois do rompimento.

Mais pedidos deste capítulo (entender e aplicar ao seu gráfico) estão na página do capítulo, no hub do livro.

## Para ir além
1. Na Parte 2, troque o ziguezague de 5% para 3% e para 8%. Quantos OCOs a mais (ou a menos) aparecem?
2. Use a função mostra() para ver o 2º e o 3º melhor exemplo de cada padrão. Você concorda com o scanner?
3. Na Parte 9, troque 60 por 120 pregões. A máxima da base melhora?
4. Na Parte 11, ponha as suas ações na lista ACOES e rode de novo.

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.